# V3 — Kế thừa V2 và tối ưu S6


## 1. Clone repository và kiểm tra môi trường

Bật **Runtime → Change runtime type → GPU**. `REPO_URL` giống notebook V1. `BRANCH` mặc định là `v3`. Dùng nhánh đã có code crop với stack gộp hoặc dùng checkout được upload. Tên branch không quyết định version pipeline.

Repo được clone vào `/content/panorama-stitching-project`. Với checkout có sẵn và sạch, `UPDATE_REPO=True` sẽ fetch/checkout/pull nhánh đã chọn. Nếu có thay đổi local, cell giữ chúng và bỏ qua cập nhật. Đặt `UPDATE_REPO=False` để dùng project đã upload/giải nén. Dataset cần có trước cell benchmark.

Giữ cùng GPU/session, tham số và build cho mọi phép so sánh. Mỗi lần chạy lưu vào thư mục report riêng.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import json
import os
import shlex
import shutil
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO_URL = 'https://github.com/nganbee/panorama-stitching-project.git'
BRANCH = 'v3'  # Set this to the branch containing the updated V3 sources.
REPO_DIR = Path('/content/panorama-stitching-project')
UPDATE_REPO = True  # False: use existing/uploaded code without fetching or switching branches.

def run(args, check=True, log=None):
    args = [str(arg) for arg in args]
    command = shlex.join(args)
    print('$', command, flush=True)
    result = subprocess.run(args, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout, flush=True)
    if log is not None:
        Path(log).write_text('$ ' + command + '\n' + result.stdout, encoding='utf-8')
    if check:
        result.check_returncode()
    return result

if not REPO_DIR.exists():
    run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR])
elif UPDATE_REPO and (REPO_DIR / '.git').exists():
    local_changes = run(['git', '-C', REPO_DIR, 'status', '--porcelain']).stdout.strip()
    if local_changes:
        print('Using existing local changes; fetch/checkout/pull skipped.')
    else:
        run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH])
        run(['git', '-C', REPO_DIR, 'checkout', BRANCH])
        run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', BRANCH])
else:
    print('Using existing/uploaded project:', REPO_DIR)

required_sources = ['src/stage_implementations/v2_stage4.cu', 'src/cuda/v2_pipeline.cu',
                    'src/stage_implementations/v3_stage6.cu', 'src/cuda/v3_pipeline.cu',
                    'src/tools/v3/test_v3_stage6.cu', 'src/tools/v3/stage6_reference.cu',
                    'src/tools/v3/test_v3_pipeline.cpp', 'tools/v2/test_matching.cpp']
missing_sources = [name for name in required_sources if not (REPO_DIR / name).is_file()]
assert not missing_sources, ('Missing cumulative V3 sources: ' + ', '.join(missing_sources) +
                             '. Use the branch or uploaded checkout containing V2 and V3.')
os.chdir(REPO_DIR)
GIT_BRANCH = GIT_COMMIT = None
if (REPO_DIR / '.git').exists():
    GIT_BRANCH = run(['git', 'branch', '--show-current']).stdout.strip()
    GIT_COMMIT = run(['git', 'rev-parse', 'HEAD']).stdout.strip()
print('Working directory:', Path.cwd())
RUNS = 5
RUN_NSYS = True
RUN_NCU = False
RUN_SANITIZER = True
assert RUNS >= 5
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
REPORT_DIR = Path('reports/v3_experiment') / RUN_ID
OUT_DIR = Path('out/v3_experiment') / RUN_ID
for directory in [Path('build'), REPORT_DIR, OUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

assert shutil.which('nvcc'), 'CUDA toolkit required; select a Colab GPU runtime.'
run(['nvidia-smi'], log=REPORT_DIR / 'nvidia-smi.txt')
GPU = run(['nvidia-smi', '--query-gpu=name,uuid,driver_version', '--format=csv,noheader']).stdout.strip()
CUDA = run(['nvcc', '--version']).stdout
CPU_COMPILER = run(['g++', '--version']).stdout
ENVIRONMENT = {'gpu': GPU, 'cuda': CUDA, 'g++': CPU_COMPILER, 'runs': RUNS, 'warmup_runs': 1,
               'baseline': 'V2', 'candidate': 'V2 S1-S5 + V3 texture warp and compressed crop stack',
               'run_id': RUN_ID, 'repo_url': REPO_URL, 'requested_branch': BRANCH,
               'actual_branch': GIT_BRANCH, 'git_commit': GIT_COMMIT}
(REPORT_DIR / 'environment.json').write_text(json.dumps(ENVIRONMENT, indent=2), encoding='utf-8')
print('Reports:', REPORT_DIR)


## 2. Bottleneck cần kiểm chứng

Dùng timing và profiler của V2 để xác định tỷ trọng của warp và crop trong S6 sau khi đã tối ưu S4. Crop gộp giảm số lần truy cập stack bằng cách lưu chung các cột cùng chiều cao. Benchmark kiểm tra đủ 12 cặp ảnh với V2 làm baseline của V3.


## 3. Ý tưởng và implementation

V2 dùng S1–S3 và S5–S6 của V1, thay S4 bằng descriptor tiling trong shared memory (`v2_match`). V3 dùng chung luồng S1–S5 của V2 và chọn `v3_warp_blend` cho S6.

S6 V3 dùng linear texture để đọc RGB với `tex1Dfetch<unsigned char>`, giữ nguyên bilinear thủ công bằng `double` và feather. Khi quét histogram của từng hàng, các cột cùng chiều cao đang hoạt động dùng chung một mục `(height, left)`. Cột có chiều cao 0 không chiếm chỗ trong stack, và mục trên cùng giữ trong biến cục bộ trong lúc quét. Crop dùng cùng quy tắc chọn hình chữ nhật như V1 và áp dụng cho mọi kích thước ảnh.

Notebook build một chương trình nhận `--version 0`, `1`, `2` hoặc `3` từ cùng checkout. V2 và V3 cùng chạy S4 đã tối ưu của V2.


In [ ]:
INCLUDES = ['-Isrc/core_headers', '-Isrc/libraries']
CUDA_FLAGS = ['-O2', '-std=c++17', '--fmad=false', '-DPANORAMA_WITH_CUDA']
V3_FLAGS = CUDA_FLAGS + ['-DPANORAMA_WITH_V2', '-DPANORAMA_WITH_V3']
V0_COMMON = ['src/integration/v0_pipeline.cpp',
             'src/stage_implementations/v0_stage12.cpp', 'src/stage_implementations/v0_stage34.cpp',
             'src/stage_implementations/v0_stage5.cpp', 'src/stage_implementations/v0_stage6.cpp']
V1_SOURCES = V0_COMMON + ['src/stage_implementations/v1_stage12.cu',
             'src/stage_implementations/v1_stage34.cu', 'src/stage_implementations/v1_stage5.cu',
             'src/stage_implementations/v1_stage6.cu', 'src/cuda/cuda_pipeline.cu',
             'src/libraries/stb_impl.cpp']
V2_SOURCES = V1_SOURCES + ['src/cuda/v2_pipeline.cu', 'src/stage_implementations/v2_stage4.cu']
V3_SOURCES = V2_SOURCES + ['src/cuda/v3_pipeline.cu', 'src/stage_implementations/v3_stage6.cu']
S6_SOURCES = ['src/stage_implementations/v0_stage6.cpp',
              'src/stage_implementations/v1_stage6.cu', 'src/stage_implementations/v3_stage6.cu']
BINARY = 'build/stitch_v3'
VERSIONS = [0, 1, 2, 3]
run(['nvcc', *V3_FLAGS, *INCLUDES, 'src/integration/main.cpp', *V3_SOURCES, '-o', BINARY])
run(['nvcc', *V3_FLAGS, *INCLUDES, 'tools/v2/test_matching.cpp',
     'src/stage_implementations/v0_stage34.cpp', 'src/stage_implementations/v1_stage34.cu',
     'src/stage_implementations/v2_stage4.cu', '-o', 'build/stage34_v2_selftest'])
run(['nvcc', *V3_FLAGS, *INCLUDES, 'src/tools/v3/test_v3_stage6.cu',
     'src/tools/v3/stage6_reference.cu', *S6_SOURCES, '-o', 'build/test_v3_stage6'])
run(['nvcc', *V3_FLAGS, *INCLUDES, 'src/tools/v3/test_v3_pipeline.cpp', *V3_SOURCES,
     '-o', 'build/test_v3_pipeline'])
print('GPU:', GPU)
versions = run([BINARY, '--list-versions']).stdout.split()
assert versions == list(map(str, VERSIONS)), versions
parameters = json.loads(run([BINARY, '--dump-params']).stdout)
assert parameters['available_versions'] == VERSIONS
(REPORT_DIR / 'parameters.json').write_text(json.dumps(parameters, indent=2), encoding='utf-8')
ENVIRONMENT['build_flags'] = V3_FLAGS
ENVIRONMENT['sources'] = V3_SOURCES
ENVIRONMENT['binary'] = BINARY
(REPORT_DIR / 'environment.json').write_text(json.dumps(ENVIRONMENT, indent=2), encoding='utf-8')


## 4. Correctness và ca lỗi

Test riêng S4 kiểm tra matching của V2 với V0/V1. Test S6 dùng S6 của V1 làm mốc cho canvas, mask, tọa độ crop và RGB vì V2 dùng lại S6 đó. Test pipeline chạy V0–V3, so với V0 và kiểm tra V2/V3 có cùng thứ tự matches, homography và panorama trùng từng byte. Ca không tạo được panorama được ghi là chưa xác nhận.


In [ ]:
run([BINARY, '--selftest', '--version', '1'], log=REPORT_DIR / 'smoke_v1.txt')
run(['build/stage34_v2_selftest'], log=REPORT_DIR / 'stage4_v2_correctness.txt')
run([BINARY, '--selftest', '--version', '3'], log=REPORT_DIR / 'smoke_v3.txt')
stage6_check = run(['build/test_v3_stage6'], log=REPORT_DIR / 'stage6_correctness.txt')
assert 'PASS: V3 S6 texture and compressed crop stack verification' in stage6_check.stdout, (
    'Use the checkout containing the compressed crop stack implementation and tests.')
run(['build/test_v3_pipeline'], log=REPORT_DIR / 'pipeline_synthetic.txt')
if RUN_SANITIZER and shutil.which('compute-sanitizer'):
    run(['compute-sanitizer', '--tool', 'memcheck', '--error-exitcode', '1', 'build/test_v3_stage6'],
        log=REPORT_DIR / 'sanitizer_memcheck.txt')
else:
    print('Compute Sanitizer not run; not counted as a memory-check PASS.')


## 5. Timing và correctness trên dataset

Chạy V0–V3 trên đủ 4 cảnh × 3 độ phân giải. Thứ tự V2/V3 được đảo luân phiên theo cảnh để cân bằng trong từng độ phân giải. CLI làm nóng một lần và lấy trung bình ít nhất năm lượt. End-to-end gồm cấp phát, tạo texture, launch, đồng bộ và copy. I/O ảnh nằm ngoài khoảng đo. Mỗi kết quả lưu GPU, ca, phiên bản và report gốc.


In [ ]:
SCENES = ['location1', 'location2', 'location3', 'challenge']
RESOLUTIONS = ['small_720p', 'medium_1080p', 'large_4k']
CASES = [(scene, resolution, Path('dataset') / scene / resolution / '1.jpg',
          Path('dataset') / scene / resolution / '2.jpg')
         for scene in SCENES for resolution in RESOLUTIONS]
missing = [str(path) for _, _, left, right in CASES for path in [left, right] if not path.is_file()]
assert not missing, 'Missing dataset inputs: ' + ', '.join(missing)
records = []
for case_index, (scene, resolution, left, right) in enumerate(CASES):
    case = scene + '_' + resolution
    print('\nCASE:', case, 'GPU:', GPU, flush=True)
    verification = run(['build/test_v3_pipeline', left, right], check=False,
                       log=REPORT_DIR / (case + '_correctness.txt'))
    parity = 'PASS' if verification.returncode == 0 else 'FAILED/UNVERIFIED'
    versions = VERSIONS if (case_index // len(RESOLUTIONS)) % 2 == 0 else [0, 1, 3, 2]
    for version in versions:
        print('GPU:', GPU, 'case:', case, 'version:', version, flush=True)
        report = REPORT_DIR / (case + '_v' + str(version) + '.json')
        output = OUT_DIR / (case + '_v' + str(version) + '.png')
        result = run([BINARY, left, right, output, '--version', version, '--runs', RUNS, '--json', report],
                     check=False, log=REPORT_DIR / (case + '_v' + str(version) + '.txt'))
        metrics = json.loads(report.read_text(encoding='utf-8')) if report.is_file() else {}
        row = {'case': case, 'scene': scene, 'resolution': resolution, 'version': version,
               'gpu': GPU, 'baseline': 'V2', 'parity': parity,
               'status': 'ok' if result.returncode == 0 and metrics.get('status') == 'ok' else 'failed',
               'exit_code': result.returncode, 'report': str(report)}
        row.update(metrics.get('timings_ms', {}))
        for stage, value in metrics.get('stage_timings_ms', {}).items():
            row['stage_' + stage] = value
        records.append(row)
    pd.DataFrame(records).to_csv(REPORT_DIR / 'timings.csv', index=False)
results = pd.DataFrame(records)
display(results)
print('Verified panorama pairs:', results.loc[results.parity.eq('PASS'), 'case'].nunique(), '/', len(CASES))
print('Full 12-case run:', len(CASES) == 12)
display(results.loc[results.status.ne('ok') | results.parity.ne('PASS')])


## 6. Result và analysis

Chỉ tính speedup khi pipeline của cả bốn phiên bản chạy thành công và kiểm tra đầu ra PASS. Cột S6 cộng tất cả kernel của stage này. V2/V3 đo hiệu quả của toàn bộ thay đổi S6 khi cả hai bản dùng cùng S4. Dùng profiler để xem riêng warp và crop.


In [ ]:
summary_rows = []
for case, group in results.groupby('case', sort=False):
    by_version = {int(row['version']): row for _, row in group.iterrows()}
    if not all(v in by_version and by_version[v]['status'] == 'ok' and by_version[v]['parity'] == 'PASS'
               for v in VERSIONS):
        continue
    v0, v1, v2, v3 = [by_version[v] for v in VERSIONS]
    summary_rows.append({'case': case, 'GPU': GPU,
        'S4 V1 (ms)': v1['match'], 'S4 V2 (ms)': v2['match'], 'S4 V3 (ms)': v3['match'],
        'S6 V2 (ms)': v2['warp_blend'], 'S6 V3 (ms)': v3['warp_blend'],
        'S6 speedup V2/V3': v2['warp_blend'] / v3['warp_blend'],
        'E2E V0 (ms)': v0['end_to_end'], 'E2E V1 (ms)': v1['end_to_end'],
        'E2E V2 (ms)': v2['end_to_end'], 'E2E V3 (ms)': v3['end_to_end'],
        'E2E speedup V0/V1': v0['end_to_end'] / v1['end_to_end'],
        'E2E speedup V0/V2': v0['end_to_end'] / v2['end_to_end'],
        'E2E speedup V1/V2': v1['end_to_end'] / v2['end_to_end'],
        'E2E speedup V2/V3': v2['end_to_end'] / v3['end_to_end'],
        'E2E speedup V0/V3': v0['end_to_end'] / v3['end_to_end'],
        'V3 kernels (ms)': v3['kernel_total'], 'V3 H2D (ms)': v3['h2d'], 'V3 D2H (ms)': v3['d2h']})
summary = pd.DataFrame(summary_rows)
summary.to_csv(REPORT_DIR / 'comparison.csv', index=False)
display(summary)
if not summary.empty:
    ax = summary.set_index('case')[['S6 V2 (ms)', 'S6 V3 (ms)']].plot.bar(figsize=(13, 4))
    ax.set_ylabel('CUDA-event kernel time (ms)')
    ax.set_title('S6 V2 vs V3 (texture warp + compressed crop stack)')
    plt.tight_layout()
    plt.show()
    ax = summary.set_index('case')[['E2E V2 (ms)', 'E2E V3 (ms)']].plot.bar(figsize=(13, 4))
    ax.set_ylabel('Host wall-clock time (ms), excluding image I/O')
    plt.tight_layout()
    plt.show()
else:
    print('No verified successful comparison; investigate logs. No speedup claim.')


## 7. Profiling evidence

Profile V2 và V3 ở cùng ca. Cả hai trace chạy `nearest_tiled_kernel` ở S4. So kernel warp, kernel quét crop và tổng các kernel crop ở S6 để biết từng phần thay đổi bao nhiêu. Cell ghi lại trạng thái của profiler.


In [ ]:
PROFILE_CASE = ('location1', 'medium_1080p')
scene, resolution = PROFILE_CASE
left = Path('dataset') / scene / resolution / '1.jpg'
right = left.with_name('2.jpg')
profile_status = []
for version in [2, 3]:
    target = [BINARY, left, right, OUT_DIR / ('profile_v' + str(version) + '.png'),
              '--version', version, '--runs', RUNS]
    print('PROFILE GPU:', GPU, 'case:', PROFILE_CASE, 'version:', version)
    if RUN_NSYS and shutil.which('nsys'):
        prefix = REPORT_DIR / ('nsys_v' + str(version))
        trace = run(['nsys', 'profile', '--trace=cuda', '--sample=none', '--force-overwrite=true',
                     '-o', prefix, *target], check=False, log=prefix.with_suffix('.log'))
        report = prefix.with_suffix('.nsys-rep')
        if trace.returncode == 0 and report.is_file():
            stats = run(['nsys', 'stats', report], check=False,
                        log=REPORT_DIR / ('nsys_v' + str(version) + '_stats.txt'))
            profile_status.append({'version': version, 'tool': 'nsys', 'status': 'captured' if stats.returncode == 0 else 'stats failed'})
        else:
            profile_status.append({'version': version, 'tool': 'nsys', 'status': 'failed; inspect log'})
    else:
        profile_status.append({'version': version, 'tool': 'nsys', 'status': 'not run / unavailable'})
    if RUN_NCU and shutil.which('ncu'):
        prefix = REPORT_DIR / ('ncu_v' + str(version))
        counters = run(['ncu', '--set', 'basic', '--kernel-name', 'regex:warp_blend_(texture_)?kernel',
                        '--force-overwrite', '-o', prefix, *target], check=False,
                       log=prefix.with_suffix('.txt'))
        profile_status.append({'version': version, 'tool': 'ncu', 'status': 'captured' if counters.returncode == 0 else 'failed; inspect log'})
    else:
        profile_status.append({'version': version, 'tool': 'ncu', 'status': 'not run / unavailable'})
pd.DataFrame(profile_status).to_csv(REPORT_DIR / 'profiling_status.csv', index=False)
display(pd.DataFrame(profile_status))


## 8. Diễn giải kết quả V2 → V3

- Báo chênh lệch S6 và end-to-end của V3 so với V2 trên từng độ phân giải.
- Dùng trace để phân tích phần chênh lệch đến từ warp hay crop. Muốn đo riêng crop, hai bản cần cùng warp và chỉ khác cách quét crop.
- Lưu notebook cùng output GPU và thư mục reports của lần đo mới.
